# Create NOMIS Foundation Awards (GRANT PATTERN, method-2 WordPress REST API)

NOMIS Foundation research projects from the foundation's own WordPress REST API at nomisfoundation.ch. Method-2 ladder (REST/GraphQL API) — clean public endpoints, no auth, no pagination quirks.

**Prerequisites:**
- Run `scripts/local/nomis_foundation_to_s3.py` first.

**Data source:** https://nomisfoundation.ch/wp-json/wp/v2/projects (paginated, ~101 records) + https://nomisfoundation.ch/wp-json/wp/v2/organization (taxonomy with ~99 institutions)
**S3 location:** `s3a://openalex-ingest/awards/nomis/nomis_projects.parquet`

**Awarding body:**
- funder_id: 4320325162
- display_name: "NOMIS Stiftung"
- country: CH (Switzerland)
- ROR: none
- DOI: 10.13039/501100008483

NOMIS Foundation is a Swiss basic-research funder supporting projects in human behavior, cognitive science, biology, archaeology, philosophy of science.

**Coverage (from 2026-05-22 local `--skip-upload` run):**
- 101 project rows; 99 distinct partner organizations indexed
- 100% title / slug / description / primary_organization / award_year
- 101/101 unique funder_award_id (`nomis-{slug}`)
- Year range: 2004 (single) and 2012-2026 contiguous, with peaks 2018 (13), 2024 (11), 2021 (11)
- Top organizations: University of Zurich (9), Salk Institute (6), ETH Zurich (5), Stanford Medicine (4), Yale (3), Caltech (3)

**Amount/currency:** NULL with §6.7 waiver. NOMIS publishes program-level funding philosophy on its /how-we-give/ page but does NOT publish per-project amounts in the API or on rendered project pages. Per the runbook's source-authority rule we don't backfill from external press releases or third-party databases. Grant/fellowship-pattern precedent for NULL amount: HHMI #44, CIFAR #79, Damon Runyon #73, Packard #95, Rita Allen #107, Schmidt Sciences #108.

**Lead investigator:** PI names are embedded in the project content as prose (e.g., "Nicholas A. Christakis' lab"). The MVP does not extract them — `lead_investigator` is left NULL. A follow-up could enrich via the `/wp/v2/person` endpoint (207 person records) if a project↔person linkage table is discoverable. The `primary_organization` field is populated 100% from the structured `organization` taxonomy.

**Provenance:** `nomis_projects` (direct from foundation, not an aggregator).

**Priority:** 109 (UCOP 106, Rita Allen 107, Schmidt Sciences 108 are immediately-prior slots in flight; Vilcek at 105 is the current main registry head).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nomis_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nomis/nomis_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.nomis_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.nomis_raw;


## Step 1.5: Inspect Raw Data, Money Scan, Native Key

NOMIS is non-monetary in our schema (§6.7 waiver). The runbook §1.5 money-field discovery scan is run for completeness; we expect zero matches because the parquet was built without monetary columns.

In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.nomis_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.nomis_raw)
WHERE LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT
    project_id, slug, title, award_year, primary_organization,
    SUBSTR(description, 1, 200) AS desc_preview
FROM openalex.awards.nomis_raw LIMIT 5;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.nomis_raw;


In [ ]:
%sql
-- Year distribution. Expected: 2004 (1) + 2012-2026 contiguous, peaks 2018 (13), 2024 (11), 2021 (11)
SELECT award_year, COUNT(*) as projects
FROM openalex.awards.nomis_raw
WHERE award_year IS NOT NULL
GROUP BY award_year ORDER BY award_year;


In [ ]:
%sql
-- Top organizations
SELECT primary_organization, COUNT(*) as projects
FROM openalex.awards.nomis_raw
GROUP BY primary_organization
ORDER BY projects DESC LIMIT 15;


In [ ]:
%sql
-- Multi-org projects: how many list more than 1 organization?
SELECT
    CASE
        WHEN organization_names IS NULL THEN 'no org'
        WHEN organization_names LIKE '%[%]' AND organization_names NOT LIKE '%,%' THEN 'single org'
        ELSE 'multi org'
    END AS shape,
    COUNT(*) AS projects
FROM openalex.awards.nomis_raw
GROUP BY shape ORDER BY projects DESC;


## Step 1.6: Fail-fast — Verify the NOMIS Funder Row Exists

The Step 2 transform CROSS JOINs against `openalex.common.funder`. **Runbook §2.2.4 mandatory.**

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320325162;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nomis_awards
USING delta
AS
WITH
nomis_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320325162
),

raw_prepared AS (
    SELECT
        *,
        TRY_CAST(award_year AS INT) AS parsed_year,
        TRY_TO_DATE(CONCAT(award_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date
    FROM openalex.awards.nomis_raw
    WHERE title IS NOT NULL
      AND TRIM(title) != ''
      AND funder_award_id IS NOT NULL
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.title as display_name,
        g.description as description,

        f.funder_id,
        g.funder_award_id,

        CAST(NULL AS DOUBLE) as amount,    -- §6.7 waiver
        CAST(NULL AS STRING) as currency,  -- §6.7 waiver

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'research' as funding_type,
        'NOMIS Research Project' as funder_scheme,

        'nomis_projects' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        g.parsed_year as start_year,
        CAST(NULL AS INT) as end_year,

        -- lead_investigator is NULL for the MVP (PI names embedded in content prose,
        -- not in structured taxonomy). The primary_organization affiliation is captured
        -- in funder_scheme metadata so downstream queries can still group by institution.
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.link as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN nomis_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 109

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nomis_projects' AND priority = 109;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    109 as priority  -- NOMIS priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.nomis_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.nomis_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.nomis_awards;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(landing_page_url) as has_url,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) as pct_description,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date,
    ROUND(COUNT(landing_page_url) * 100.0 / COUNT(*), 1) as pct_url
FROM openalex.awards.nomis_awards;


In [ ]:
%sql
SELECT
    funder_award_id, display_name, start_year, funder_scheme,
    landing_page_url, SUBSTR(description, 1, 120) AS desc_preview
FROM openalex.awards.nomis_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.nomis_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.nomis_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 25;


In [ ]:
%sql
-- §6.7 Amount/currency — intentionally NULL by waiver
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.nomis_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.nomis_awards;
